# Tahap 6 — Quality Assessment

**Tujuan notebook**
Melakukan audit kualitas data murni (read-only) terhadap `05_integrated_dataset.csv`
hasil Tahap 5, tanpa mengubah dataset dalam bentuk apa pun.

**Batasan cakupan (scope) — Tahap 6 TIDAK mencakup:**
- imputasi
- interpolasi
- filling missing value
- outlier removal
- labeling
- feature engineering
- scaling
- train/test split

Notebook ini hanya membaca dataset dan menghasilkan dua deliverable audit:
- `06_quality_assessment_report.md`
- `quality_issue_log.csv`


## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
import hashlib
from pathlib import Path


## 2. Konfigurasi

In [2]:
INPUT_PATH = Path("05_integrated_dataset.csv")
OUTPUT_REPORT_PATH = Path("06_quality_assessment_report.md")
OUTPUT_ISSUE_LOG_PATH = Path("quality_issue_log.csv")

SOUNDERPY_VARS = ["cin", "kindex", "li", "tt", "sweat", "cape"]
OGIMET_VARS = ["rr", "tavg", "rh"]
NUMERIC_VARS = OGIMET_VARS + SOUNDERPY_VARS


## 3. Load Dataset (Read-Only)

Dataset dimuat apa adanya, tanpa transformasi. Hash MD5 berkas mentah dicatat di awal dan
di akhir notebook untuk membuktikan berkas input tidak pernah ditulis ulang oleh proses
audit ini.

In [3]:
def file_md5(path: Path) -> str:
    """Hitung MD5 sebuah berkas untuk memverifikasi berkas tidak berubah."""
    return hashlib.md5(path.read_bytes()).hexdigest()


input_md5_before = file_md5(INPUT_PATH)

df = pd.read_csv(INPUT_PATH)
df["date"] = pd.to_datetime(df["date"], errors="raise")

print(f"Shape   : {df.shape}")
print(f"MD5 awal: {input_md5_before}")
df.head()


Shape   : (2922, 12)
MD5 awal: bbf33eba2edf05b2695edb3cb2ce3676


,date,selected_hour,selection_status,rr,tavg,rh,cin,kindex,li,tt,sweat,cape
0,2017-01-01,12Z,SELECTED,NaN,28.2,83.9,-4.906,34.4,-4.510,42.2,219.162,2226.401
1,2017-01-02,12Z,SELECTED,26.9,26.5,87.4,-22.887,35.2,-2.955,41.5,223.381,921.164
2,2017-01-03,12Z,SELECTED,78.0,26.2,87.6,0.000,36.4,-3.317,40.5,277.368,1671.572
3,2017-01-04,12Z,SELECTED,33.0,24.7,92.4,-4.865,35.1,-0.752,39.0,294.560,397.409
4,2017-01-05,12Z,SELECTED,83.0,25.7,88.3,-15.811,38.0,-5.124,44.1,299.131,2064.721


## 4. Task 1 — Audit Struktur Dataset

In [4]:
def audit_structure(df: pd.DataFrame) -> dict:
    """Audit struktural dasar: jumlah baris/kolom, duplicate date, duplicate row."""
    return {
        "jumlah_row": len(df),
        "jumlah_kolom": len(df.columns),
        "duplicate_date": int(df["date"].duplicated().sum()),
        "duplicate_row": int(df.duplicated().sum()),
    }


structure_audit = audit_structure(df)
structure_audit


{'jumlah_row': 2922,
 'jumlah_kolom': 12,
 'duplicate_date': 0,
 'duplicate_row': 0}

## 5. Task 2 — Audit Missing Value per Kolom

In [5]:
def audit_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Jumlah dan persentase missing value untuk setiap kolom."""
    n_total = len(df)
    n_missing = df.isna().sum()
    pct_missing = (n_missing / n_total * 100).round(2)
    return pd.DataFrame({
        "kolom": df.columns,
        "jumlah_missing": n_missing.values,
        "persentase_missing": pct_missing.values,
    })


missing_audit_df = audit_missing(df)
missing_audit_df


,kolom,jumlah_missing,persentase_missing
0,date,0,0.00
1,selected_hour,0,0.00
2,selection_status,0,0.00
3,rr,37,1.27
4,tavg,31,1.06
5,rh,17,0.58
6,cin,149,5.10
7,kindex,194,6.64
8,li,194,6.64
9,tt,194,6.64


## 6. Task 3 — Pemisahan Missing: NO_SOUNDING vs SELECTED

Untuk setiap kolom, missing dipecah menjadi:
- **A. Missing karena NO_SOUNDING** — baris dengan `selection_status = NO_SOUNDING`
- **B. Missing meskipun SELECTED** — baris dengan `selection_status = SELECTED` tetapi
  nilai kolom tetap NaN (anomali yang perlu ditelusuri lebih lanjut)

Pemisahan ini paling bermakna untuk variabel SounderPy (yang missing-nya berkaitan
langsung dengan ketersediaan sounding), namun dihitung untuk seluruh kolom numerik agar
audit tetap lengkap.

In [6]:
def audit_missing_by_selection_status(df: pd.DataFrame, variables: list) -> pd.DataFrame:
    """Pecah jumlah missing per kolom menjadi missing pada NO_SOUNDING vs SELECTED."""
    is_no_sounding = df["selection_status"] == "NO_SOUNDING"
    is_selected = df["selection_status"] == "SELECTED"

    rows = []
    for col in variables:
        is_missing = df[col].isna()
        rows.append({
            "kolom": col,
            "missing_karena_no_sounding": int((is_missing & is_no_sounding).sum()),
            "missing_meskipun_selected": int((is_missing & is_selected).sum()),
        })
    return pd.DataFrame(rows)


missing_split_df = audit_missing_by_selection_status(df, NUMERIC_VARS)
missing_split_df


,kolom,missing_karena_no_sounding,missing_meskipun_selected
0,rr,21,16
1,tavg,20,11
2,rh,17,0
3,cin,148,1
4,kindex,148,46
5,li,148,46
6,tt,148,46
7,sweat,148,46
8,cape,148,1


## 7. Task 4 — Tanggal SELECTED tetapi Variabel Atmosfer Missing

Mengidentifikasi tanggal dengan `selection_status = SELECTED` namun salah satu dari
`cin, kindex, li, tt, sweat, cape` bernilai NaN — indikasi missing pada level sumber data
mentah SounderPy, bukan akibat tidak adanya sounding.

In [7]:
def find_selected_but_missing(df: pd.DataFrame, variables: list) -> pd.DataFrame:
    """Cari baris SELECTED yang tetap memiliki NaN pada salah satu variabel atmosfer."""
    is_selected = df["selection_status"] == "SELECTED"
    records = []
    for col in variables:
        subset = df.loc[is_selected & df[col].isna(), ["date"]].copy()
        subset["variable"] = col
        records.append(subset)
    result = pd.concat(records, ignore_index=True) if records else pd.DataFrame(columns=["date", "variable"])
    return result.sort_values(["date", "variable"]).reset_index(drop=True)


selected_but_missing_df = find_selected_but_missing(df, SOUNDERPY_VARS)
print(f"Jumlah kejadian (tanggal x variabel): {len(selected_but_missing_df)}")
print(f"Jumlah tanggal unik yang terdampak  : {selected_but_missing_df['date'].nunique()}")
selected_but_missing_df.head(10)


Jumlah kejadian (tanggal x variabel): 186
Jumlah tanggal unik yang terdampak  : 46


,date,variable
0,2017-03-30,kindex
1,2017-03-30,li
2,2017-03-30,sweat
3,2017-03-30,tt
4,2017-03-31,kindex
5,2017-03-31,li
6,2017-03-31,sweat
7,2017-03-31,tt
8,2017-04-13,kindex
9,2017-04-13,li


## 8. Task 5 — Audit Rentang Nilai

Min, max, mean, median untuk seluruh variabel numerik.

In [8]:
def audit_value_range(df: pd.DataFrame, variables: list) -> pd.DataFrame:
    """Statistik rentang nilai dasar untuk setiap variabel numerik."""
    rows = []
    for col in variables:
        series = df[col].dropna()
        rows.append({
            "kolom": col,
            "min": series.min(),
            "max": series.max(),
            "mean": round(series.mean(), 3),
            "median": series.median(),
        })
    return pd.DataFrame(rows)


value_range_df = audit_value_range(df, NUMERIC_VARS)
value_range_df


,kolom,min,max,mean,median
0,rr,0.000,355.300,12.403,1.400
1,tavg,23.700,30.700,26.966,27.000
2,rh,66.600,97.000,85.328,85.400
3,cin,-515.072,0.000,-49.488,-22.121
4,kindex,-2.700,134.081,33.494,34.400
5,li,-67.627,24.926,-3.610,-3.929
6,tt,24.000,168.700,43.221,43.400
7,sweat,27.214,2625.196,209.753,212.478
8,cape,0.000,38045.450,1528.955,1504.458


## 9. Task 6 — Audit Outlier (Metode IQR)

Tidak ada penghapusan data. Outlier hanya diidentifikasi menggunakan pagar IQR standar:
batas bawah `Q1 - 1.5*IQR`, batas atas `Q3 + 1.5*IQR`.

In [9]:
def audit_outliers_iqr(df: pd.DataFrame, variables: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Identifikasi outlier per variabel numerik dengan metode IQR (tanpa menghapus data)."""
    summary_rows = []
    detail_rows = []

    for col in variables:
        series = df[col].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        is_outlier = (df[col] < lower_bound) | (df[col] > upper_bound)
        outlier_rows = df.loc[is_outlier & df[col].notna(), ["date", col]]

        summary_rows.append({
            "kolom": col,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "batas_bawah": lower_bound,
            "batas_atas": upper_bound,
            "jumlah_outlier": len(outlier_rows),
            "nilai_ekstrem_min": outlier_rows[col].min() if len(outlier_rows) else np.nan,
            "nilai_ekstrem_max": outlier_rows[col].max() if len(outlier_rows) else np.nan,
        })

        for row in outlier_rows.itertuples(index=False):
            detail_rows.append({"date": row.date, "variable": col, "value": getattr(row, col)})

    summary_df = pd.DataFrame(summary_rows)
    detail_df = pd.DataFrame(detail_rows, columns=["date", "variable", "value"])
    return summary_df, detail_df


outlier_summary_df, outlier_detail_df = audit_outliers_iqr(df, NUMERIC_VARS)
outlier_summary_df


,kolom,q1,q3,iqr,batas_bawah,batas_atas,jumlah_outlier,nilai_ekstrem_min,nilai_ekstrem_max
0,rr,0.00000,13.00000,13.00000,-19.500000,32.500000,364,32.700,355.300
1,tavg,26.40000,27.60000,1.20000,24.600000,29.400000,29,23.700,30.700
2,rh,82.70000,88.10000,5.40000,74.600000,96.200000,15,66.600,97.000
3,cin,-56.78200,-6.70100,50.08100,-131.903500,68.420500,280,-515.072,-132.659
4,kindex,31.60000,36.80000,5.20000,23.800000,44.600000,139,-2.700,134.081
5,li,-5.13850,-2.37925,2.75925,-9.277375,1.759625,58,-67.627,24.926
6,tt,41.80000,44.80000,3.00000,37.300000,49.300000,89,24.000,168.700
7,sweat,195.84475,227.93950,32.09475,147.702625,276.081625,123,27.214,2625.196
8,cape,565.55400,2291.70100,1726.14700,-2023.666500,4880.921500,12,4970.330,38045.450


## 10. Susun `quality_issue_log.csv`

In [10]:
def build_issue_log(
    df: pd.DataFrame,
    ogimet_vars: list,
    sounderpy_vars: list,
    outlier_detail_df: pd.DataFrame,
) -> pd.DataFrame:
    """Susun log seluruh isu kualitas pada level (date, variable) menjadi satu tabel."""
    is_no_sounding = df["selection_status"] == "NO_SOUNDING"
    is_selected = df["selection_status"] == "SELECTED"

    entries = []

    # Missing pada variabel SounderPy: dipecah NO_SOUNDING vs SELECTED
    for col in sounderpy_vars:
        missing_no_sounding = df.loc[df[col].isna() & is_no_sounding, "date"]
        for d in missing_no_sounding:
            entries.append({"date": d, "variable": col, "issue_type": "MISSING_NO_SOUNDING", "value": np.nan})

        missing_selected = df.loc[df[col].isna() & is_selected, "date"]
        for d in missing_selected:
            entries.append({"date": d, "variable": col, "issue_type": "MISSING_DESPITE_SELECTED", "value": np.nan})

    # Missing pada variabel Ogimet: tidak terkait status sounding
    for col in ogimet_vars:
        missing_ogimet = df.loc[df[col].isna(), "date"]
        for d in missing_ogimet:
            entries.append({"date": d, "variable": col, "issue_type": "MISSING_VALUE", "value": np.nan})

    # Duplicate date / duplicate row (jika ada)
    for d in df.loc[df["date"].duplicated(), "date"]:
        entries.append({"date": d, "variable": "date", "issue_type": "DUPLICATE_DATE", "value": np.nan})
    for d in df.loc[df.duplicated(), "date"]:
        entries.append({"date": d, "variable": "ALL", "issue_type": "DUPLICATE_ROW", "value": np.nan})

    issue_df = pd.DataFrame(entries, columns=["date", "variable", "issue_type", "value"])

    # Outlier IQR
    outlier_entries = outlier_detail_df.copy()
    outlier_entries["issue_type"] = "OUTLIER_IQR"
    outlier_entries = outlier_entries[["date", "variable", "issue_type", "value"]]

    issue_df = pd.concat([issue_df, outlier_entries], ignore_index=True)
    issue_df = issue_df.sort_values(["date", "variable", "issue_type"]).reset_index(drop=True)
    return issue_df


quality_issue_log_df = build_issue_log(df, OGIMET_VARS, SOUNDERPY_VARS, outlier_detail_df)
print(f"Total baris quality_issue_log: {len(quality_issue_log_df)}")
quality_issue_log_df["issue_type"].value_counts()


Total baris quality_issue_log: 2268


issue_type
OUTLIER_IQR                 1109
MISSING_NO_SOUNDING          888
MISSING_DESPITE_SELECTED     186
MISSING_VALUE                 85
Name: count, dtype: int64

In [11]:
quality_issue_log_df.to_csv(OUTPUT_ISSUE_LOG_PATH, index=False)
print(f"Tersimpan: {OUTPUT_ISSUE_LOG_PATH.resolve()}")
quality_issue_log_df.head(10)


Tersimpan: /home/claude/work/quality_issue_log.csv


,date,variable,issue_type,value
0,2017-01-01,rr,MISSING_VALUE,NaN
1,2017-01-03,rr,OUTLIER_IQR,78.000
2,2017-01-03,sweat,OUTLIER_IQR,277.368
3,2017-01-04,rr,OUTLIER_IQR,33.000
4,2017-01-04,sweat,OUTLIER_IQR,294.560
5,2017-01-05,rr,OUTLIER_IQR,83.000
6,2017-01-05,sweat,OUTLIER_IQR,299.131
7,2017-01-06,rr,OUTLIER_IQR,57.000
8,2017-01-06,tt,OUTLIER_IQR,35.500
9,2017-01-07,kindex,OUTLIER_IQR,21.100


## 11. Verifikasi: Dataset Input Tidak Berubah

In [12]:
input_md5_after = file_md5(INPUT_PATH)
assert input_md5_before == input_md5_after, "Berkas input 05_integrated_dataset.csv berubah selama audit!"
assert len(df) == 2922 and len(df.columns) == 12, "Bentuk dataset di memori tidak sesuai ekspektasi input."
print("Terverifikasi: 05_integrated_dataset.csv TIDAK dimodifikasi oleh proses audit ini.")
print(f"MD5 sebelum: {input_md5_before}")
print(f"MD5 sesudah: {input_md5_after}")


Terverifikasi: 05_integrated_dataset.csv TIDAK dimodifikasi oleh proses audit ini.
MD5 sebelum: bbf33eba2edf05b2695edb3cb2ce3676
MD5 sesudah: bbf33eba2edf05b2695edb3cb2ce3676


## 12. Susun `06_quality_assessment_report.md`

In [13]:
def fmt_table(df: pd.DataFrame) -> str:
    """Format DataFrame kecil menjadi tabel Markdown sederhana."""
    return df.to_markdown(index=False)


def build_quality_report(
    structure_audit: dict,
    missing_audit_df: pd.DataFrame,
    missing_split_df: pd.DataFrame,
    selected_but_missing_df: pd.DataFrame,
    value_range_df: pd.DataFrame,
    outlier_summary_df: pd.DataFrame,
    quality_issue_log_df: pd.DataFrame,
) -> str:
    """Bangun konten 06_quality_assessment_report.md dari hasil audit aktual."""
    n_selected_but_missing_dates = selected_but_missing_df["date"].nunique()

    report = f"""# 06_quality_assessment_report — Tahap 6: Quality Assessment

Dataset yang diaudit: `05_integrated_dataset.csv` (dibaca apa adanya, tidak dimodifikasi).

## 1. Audit Struktur Dataset

- Jumlah row     : {structure_audit['jumlah_row']}
- Jumlah kolom   : {structure_audit['jumlah_kolom']}
- Duplicate date : {structure_audit['duplicate_date']}
- Duplicate row  : {structure_audit['duplicate_row']}

## 2. Audit Missing Value per Kolom

{fmt_table(missing_audit_df)}

## 3. Pemisahan Missing: NO_SOUNDING vs SELECTED

{fmt_table(missing_split_df)}

**Interpretasi:**
- `missing_karena_no_sounding` bersifat struktural (diharapkan) — konsisten dengan 148
  hari tanpa sounding pada Tahap 3–4.
- `missing_meskipun_selected` > 0 menandakan missing pada level data mentah SounderPy itu
  sendiri (bukan akibat proses integrasi), dan menjadi perhatian utama Tahap 7.

## 4. Tanggal SELECTED tetapi Variabel Atmosfer Missing

- Jumlah kejadian (kombinasi tanggal x variabel): {len(selected_but_missing_df)}
- Jumlah tanggal unik yang terdampak: {n_selected_but_missing_dates}
- Daftar lengkap tersimpan pada `quality_issue_log.csv`
  (issue_type = `MISSING_DESPITE_SELECTED`).

## 5. Audit Rentang Nilai (Variabel Numerik)

{fmt_table(value_range_df)}

## 6. Audit Outlier (Metode IQR)

{fmt_table(outlier_summary_df.drop(columns=["nilai_ekstrem_min", "nilai_ekstrem_max"]))}

Nilai ekstrem (min/max di antara outlier yang terdeteksi):

{fmt_table(outlier_summary_df[["kolom", "nilai_ekstrem_min", "nilai_ekstrem_max"]])}

Tidak ada data yang dihapus pada langkah ini. Seluruh titik outlier tercatat individual
pada `quality_issue_log.csv` (issue_type = `OUTLIER_IQR`).

## 7. Ringkasan quality_issue_log.csv

Total baris log: {len(quality_issue_log_df)}

Distribusi per issue_type:

{fmt_table(quality_issue_log_df['issue_type'].value_counts().rename_axis('issue_type').reset_index(name='jumlah'))}

## 8. Acceptance Criteria

- [x] Tidak ada modifikasi dataset (diverifikasi lewat perbandingan MD5 berkas input sebelum/sesudah)
- [x] Tidak ada record yang dihapus (jumlah row tetap {structure_audit['jumlah_row']})
- [x] Semua masalah kualitas terdokumentasi pada `quality_issue_log.csv`
- [x] Dataset input identik dengan sebelum audit
- [x] Output siap digunakan pada Tahap 7 (Missing Value Handling)

## Catatan Cakupan

Notebook ini murni melakukan audit (read-only). Tidak ada imputasi, interpolasi, filling
missing value, outlier removal, labeling, feature engineering, scaling, maupun
train/test split yang dilakukan pada tahap ini.
"""
    return report


report_content = build_quality_report(
    structure_audit,
    missing_audit_df,
    missing_split_df,
    selected_but_missing_df,
    value_range_df,
    outlier_summary_df,
    quality_issue_log_df,
)
OUTPUT_REPORT_PATH.write_text(report_content)
print(f"Tersimpan: {OUTPUT_REPORT_PATH.resolve()}")


Tersimpan: /home/claude/work/06_quality_assessment_report.md
